# W10〜W11 連敗原因分析

MomentumStrategy(threshold=3.0, window=20) / BTC/JPY 15min 足

W09: +7.42%, 勝率67%　→　W10: -2.18%, 勝率0%　→　W11: -6.17%, 勝率10%

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch

from cryptbot.data.normalizer import normalize
from cryptbot.strategies.regime import RegimeDetector, Regime
from cryptbot.strategies.momentum import MomentumStrategy

plt.rcParams['figure.figsize'] = (16, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'

DATA_DIR = ROOT / 'data' / 'btc_jpy' / '15min'

WINDOWS = {
    'W09': ('2025-03-01', '2025-07-01'),
    'W10': ('2025-07-01', '2025-11-01'),
    'W11': ('2025-11-01', '2026-03-01'),
    'W12': ('2026-03-01', '2026-04-20'),
}

WINDOW_COLORS = {
    'W09': '#2196F3',
    'W10': '#F44336',
    'W11': '#FF9800',
    'W12': '#4CAF50',
}

THRESHOLD = 3.0
MOMENTUM_WINDOW = 20

print('Setup complete')

In [ ]:
# Section 1: データロード
def load_window_data(year_files: list[str]) -> pd.DataFrame:
    """複数 parquet を結合して normalize() 済み DataFrame を返す。"""
    frames = []
    for f in year_files:
        path = DATA_DIR / f
        if path.exists():
            frames.append(pd.read_parquet(path))
    df = pd.concat(frames).sort_values("timestamp").reset_index(drop=True)
    return normalize(df, momentum_window=MOMENTUM_WINDOW)

# W09〜W12 をカバーする年ファイル
raw_full = load_window_data(["2025.parquet", "2026.parquet"])

# 窓ごとにスライス
def slice_window(df: pd.DataFrame, start: str, end: str) -> pd.DataFrame:
    mask = (df["timestamp"] >= pd.Timestamp(start, tz="Asia/Tokyo")) & (df["timestamp"] < pd.Timestamp(end, tz="Asia/Tokyo"))
    return df[mask].copy()

window_data = {
    name: slice_window(raw_full, s, e)
    for name, (s, e) in WINDOWS.items()
}

print("=== データ確認 ===")
for name, df in window_data.items():
    print(f"{name}: {len(df):>5} bars  "
          f"{df['timestamp'].iloc[0]:%Y-%m-%d} 〜 {df['timestamp'].iloc[-1]:%Y-%m-%d}  "
          f"NaN rows: {df['momentum'].isna().sum()}")

In [ ]:
# Section 2: 価格 + レジーム俯瞰

detector = RegimeDetector()

def detect_regimes(df: pd.DataFrame) -> pd.Series:
    """各バーのレジームを計算する（rolling 20バー以上のデータを使用）。"""
    regimes = []
    for i in range(len(df)):
        window = df.iloc[max(0, i - 49): i + 1]  # 最大50バーを使用
        if len(window) < 20:
            regimes.append(None)
            continue
        try:
            regimes.append(detector.detect(window).value)
        except Exception:
            regimes.append(None)
    return pd.Series(regimes, index=df.index)

# 各窓のレジームを計算
for name, df in window_data.items():
    window_data[name] = df.copy()
    window_data[name]["regime"] = detect_regimes(df)
    print(f"{name} レジーム計算完了")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(18, 10), gridspec_kw={"height_ratios": [3, 1]})

ax_price = axes[0]
ax_regime = axes[1]

regime_palette = {
    "TREND_UP": "#4CAF50",
    "TREND_DOWN": "#F44336",
    "RANGE": "#9E9E9E",
    "HIGH_VOL": "#FF9800",
}

# 価格チャート（窓ごとに色分け）
for name, df in window_data.items():
    ax_price.plot(
        df["timestamp"], df["close"],
        color=WINDOW_COLORS[name], linewidth=1.0, label=name, alpha=0.9
    )

ax_price.set_title("BTC/JPY 終値 (W09〜W12 色分け)", fontsize=13)
ax_price.set_ylabel("Close (JPY)")
ax_price.legend(loc="upper left")
ax_price.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax_price.grid(alpha=0.3)

# レジーム分布（窓ごと積み上げ棒グラフ）
regime_order = ["TREND_UP", "TREND_DOWN", "RANGE", "HIGH_VOL"]
regime_counts = {}
for name, df in window_data.items():
    counts = df["regime"].value_counts(normalize=True) * 100
    regime_counts[name] = {r: counts.get(r, 0) for r in regime_order}

bottoms = {name: 0 for name in WINDOWS}
for regime in regime_order:
    vals = [regime_counts[name][regime] for name in WINDOWS]
    ax_regime.bar(
        list(WINDOWS.keys()), vals,
        bottom=[bottoms[n] for n in WINDOWS],
        color=regime_palette[regime], label=regime, alpha=0.85
    )
    for name, v in zip(WINDOWS, vals):
        bottoms[name] += v

ax_regime.set_ylabel("割合 (%)")
ax_regime.set_title("レジーム分布 (窓別)")
ax_regime.legend(loc="upper right", fontsize=9)
ax_regime.set_ylim(0, 105)

plt.tight_layout()
plt.savefig(ROOT / "notebooks" / "fig_price_regime.png", dpi=120)
plt.show()

print("
=== レジーム分布 (%) ===")
header = f"{'Regime':<15}" + "".join(f"{n:>8}" for n in WINDOWS)
print(header)
for regime in regime_order:
    row = f"{regime:<15}" + "".join(f"{regime_counts[n][regime]:>7.1f}%" for n in WINDOWS)
    print(row)

In [ ]:
# Section 3: ボラティリティ分析

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

ax_atr = axes[0]
ax_mom = axes[1]

# ATR rolling median (20本)
for name, df in window_data.items():
    atr_med = df["atr"].rolling(20).median()
    ax_atr.plot(df["timestamp"], atr_med,
                color=WINDOW_COLORS[name], linewidth=1.2, label=name)

ax_atr.set_title("ATR Rolling Median (20本) の推移")
ax_atr.set_ylabel("ATR (JPY)")
ax_atr.legend()
ax_atr.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax_atr.grid(alpha=0.3)

# モメンタム分布ヒストグラム
bins = np.linspace(-10, 10, 60)
for name, df in window_data.items():
    mom = df["momentum"].dropna()
    ax_mom.hist(mom, bins=bins, alpha=0.4, color=WINDOW_COLORS[name], label=name, density=True)

ax_mom.axvline(THRESHOLD, color="black", linestyle="--", linewidth=1.5, label=f"threshold={THRESHOLD}")
ax_mom.axvline(-THRESHOLD, color="black", linestyle="--", linewidth=1.5)
ax_mom.set_title(f"モメンタム分布 (threshold={THRESHOLD})")
ax_mom.set_xlabel("Momentum (%)")
ax_mom.set_ylabel("密度")
ax_mom.legend()
ax_mom.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(ROOT / "notebooks" / "fig_volatility.png", dpi=120)
plt.show()

# シグナル発生率（threshold 超過頻度）
print("\n=== シグナル発生率 (threshold=3.0 超過) ===")
print(f"{'窓':<6} {'ATR_median':>12} {'BUY率%':>8} {'SELL率%':>9} {'中立率%':>9} {'シグナル%':>10}")
for name, df in window_data.items():
    mom = df["momentum"].dropna()
    atr_med = df["atr"].rolling(20).median().dropna().median()
    buy_rate = (mom > THRESHOLD).mean() * 100
    sell_rate = (mom < -THRESHOLD).mean() * 100
    neutral_rate = 100 - buy_rate - sell_rate
    print(f"{name:<6} {atr_med:>12.0f} {buy_rate:>7.2f}% {sell_rate:>8.2f}% {neutral_rate:>8.2f}% {buy_rate+sell_rate:>9.2f}%")

In [ ]:
# Section 4: シグナル品質分析

strategy = MomentumStrategy(threshold=THRESHOLD)

def get_signals(df: pd.DataFrame) -> pd.Series:
    """各バーのシグナル方向 (+1=BUY, -1=SELL, 0=HOLD) を返す。"""
    results = []
    for i in range(len(df)):
        if i < MOMENTUM_WINDOW:
            results.append(0)
            continue
        window = df.iloc[: i + 1]
        sig = strategy.generate_signal(window)
        from cryptbot.strategies.base import Direction
        if sig.direction == Direction.BUY:
            results.append(1)
        elif sig.direction == Direction.SELL:
            results.append(-1)
        else:
            results.append(0)
    return pd.Series(results, index=df.index, name="signal")

# 各窓でシグナルを計算（BUY シグナルのみ、Long-only 戦略のため）
FORWARD_BARS = 4  # エントリー後4本(=1時間)の価格変化を確認

print("シグナル計算中...")
for name, df in window_data.items():
    window_data[name]["signal"] = get_signals(df)
    print(f"  {name}: BUY={(window_data[name]['signal'] == 1).sum()}, SELL={(window_data[name]['signal'] == -1).sum()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
ax_chart = axes[0]
ax_fwd = axes[1]

# BUY シグナルのみをチャートに overlay（W09 vs W11 比較）
for name in ["W09", "W11"]:
    df = window_data[name]
    buy_df = df[df["signal"] == 1]
    ax_chart.plot(df["timestamp"], df["close"],
                  color=WINDOW_COLORS[name], linewidth=0.8, alpha=0.7, label=f"{name} close")
    ax_chart.scatter(buy_df["timestamp"], buy_df["close"],
                     color=WINDOW_COLORS[name], marker="^", s=40, zorder=5)

ax_chart.set_title("BUY シグナル発生位置 (W09 vs W11)")
ax_chart.set_ylabel("Close (JPY)")
ax_chart.legend()
ax_chart.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax_chart.grid(alpha=0.3)

# エントリー後 FORWARD_BARS 本のリターン分布
def calc_forward_returns(df: pd.DataFrame, forward_bars: int = 4) -> pd.Series:
    """BUY シグナル発生バーから forward_bars 本後の価格変化率 (%)。"""
    buy_idx = df.index[df["signal"] == 1].tolist()
    returns = []
    closes = df["close"].values
    local_idxs = [df.index.get_loc(i) for i in buy_idx]
    for loc in local_idxs:
        if loc + forward_bars < len(closes):
            entry = closes[loc + 1]   # 翌バー約定
            exit_ = closes[loc + forward_bars + 1]
            if entry > 0:
                returns.append((exit_ - entry) / entry * 100)
    return pd.Series(returns)

fwd_returns = {name: calc_forward_returns(df, FORWARD_BARS) for name, df in window_data.items()}

for name, ret in fwd_returns.items():
    if len(ret) == 0:
        continue
    ax_fwd.hist(ret, bins=20, alpha=0.45, color=WINDOW_COLORS[name],
                label=f"{name} (n={len(ret)}, mean={ret.mean():.2f}%)")

ax_fwd.axvline(0, color="black", linewidth=1.0)
ax_fwd.set_title(f"BUY エントリー後 {FORWARD_BARS} 本リターン分布")
ax_fwd.set_xlabel("Forward Return (%)")
ax_fwd.set_ylabel("Count")
ax_fwd.legend()
ax_fwd.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(ROOT / "notebooks" / "fig_signal_quality.png", dpi=120)
plt.show()

print("\n=== エントリー後リターン統計 ===")
print(f"{'窓':<6} {'n':>5} {'mean%':>8} {'median%':>9} {'勝率%':>8} {'std':>7}")
for name, ret in fwd_returns.items():
    if len(ret) == 0:
        print(f"{name:<6}  0件のシグナルなし")
        continue
    win = (ret > 0).mean() * 100
    print(f"{name:<6} {len(ret):>5} {ret.mean():>7.2f}% {ret.median():>8.2f}% {win:>7.1f}% {ret.std():>7.2f}")